# Improved Backend Logic Test

This notebook clones and improves the logic from `vlm_llm_test.ipynb` to address issues like token bloat, entity bleed, and missing line items. It serves as a testing ground before updating `app.py`.

In [1]:
import os
import json
import re
import time
import pathlib
import shutil
from openai import OpenAI
from paddleocr import PaddleOCRVL
import pandas as pd
from bs4 import BeautifulSoup

# --- Configuration ---
VLM_SERVER_URL = "http://localhost:8000/v1"
LLM_SERVER_URL = "http://localhost:8001/v1"
MODEL_NAME = "Qwen2.5-1.5B"
WORKDIR = "./"
DEMO_IMAGE = os.path.join(WORKDIR, "../invoices/demo3.jpeg")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


## 1. Pre-Processing Logic (Text Cleaning)

This function strips out noise (like `<img ...>` tags) and converts HTML tables to Markdown to help the 1.5B model's attention.

In [2]:
def html_table_to_markdown(html_content):
    """Converts HTML <table> to Markdown table using BeautifulSoup."""
    soup = BeautifulSoup(html_content, 'html.parser')
    tables = soup.find_all('table')
    
    markdown_tables = []
    for table in tables:
        rows = table.find_all('tr')
        if not rows: continue
        
        md_rows = []
        for i, row in enumerate(rows):
            cols = row.find_all(['td', 'th'])
            cols_text = [c.get_text(strip=True) for c in cols]
            md_rows.append("| " + " | ".join(cols_text) + " |")
            
            # Add separator after header
            if i == 0:
                md_rows.append("| " + " | ".join(["---"] * len(cols)) + " |")
        
        markdown_tables.append("\n".join(md_rows))
    
    # Replace original tables with markdown in the content
    # For simplicity, if multiple tables, we just return the joined markdown
    return "\n\n".join(markdown_tables) if markdown_tables else html_content

def clean_ocr_text(text):
    """Cleans OCR text: removes img tags and converts tables."""
    # 1. Remove <img ...> tags
    text = re.sub(r'<img[^>]*>', '', text)
    
    # 2. Extract <table> contents and convert to markdown
    def table_replacer(match):
        return html_table_to_markdown(match.group(0))
    
    cleaned_text = re.sub(r'<table>.*?</table>', table_replacer, text, flags=re.DOTALL)
    
    # 3. Clean up excessive whitespace
    cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)
    
    return cleaned_text.strip()

## 2. Extraction & Prompt Engineering

Setting up the new nested `SYSTEM_PROMPT` as per requirements.

In [3]:
SYSTEM_PROMPT = """You are a precise data extraction assistant specialized in financial documents.
Extract information from the provided text and return it strictly as a JSON object.

RULES:
1. DATES: All dates MUST be converted to YYYY-MM-DD format.
2. NUMBERS: Convert currency and quantities to float numbers (e.g., $1,200.50 -> 1200.50).
3. NULLS: If a field is not present in the text, use null.
4. NESTING: Follow the exact nested structure provided below to separate Vendor vs Client details.
5. LINE ITEMS: Extract every row from tables into the line_items array.

TARGET JSON SCHEMA:
{
  "document_details": {
    "document_type": "string",
    "invoice_number": "string",
    "invoice_date": "YYYY-MM-DD",
    "due_date": "YYYY-MM-DD"
  },
  "vendor_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "client_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "line_items": [
    {
      "description": "string",
      "quantity": "float",
      "unit_price": "float",
      "line_total": "float"
    }
  ],
  "financials": {
    "subtotal": "float",
    "tax_amount": "float",
    "total_amount": "float"
  }
}"""

def extract_and_combine_content(data):
    combined_content = []
    if isinstance(data, list) and data:
        if 'parsing_res_list' in data[0] and isinstance(data[0]['parsing_res_list'], list):
            for item in data[0]['parsing_res_list']:
                content = None
                if hasattr(item, 'content'):
                    content = item.content
                elif isinstance(item, dict):
                    content = item.get('content')
                
                if content is not None:
                    combined_content.append(content)
    return '\n'.join(combined_content)

## 3. Post-Processing (Validation Gate)

Checks for mathematical consistency and adds the `requires_human_review` flag.

In [4]:
def validate_extraction(data):
    """Validates the extracted JSON data for errors and mathematical consistency."""
    issues = []
    
    financials = data.get("financials", {})
    subtotal = financials.get("subtotal") or 0.0
    tax = financials.get("tax_amount") or 0.0
    total = financials.get("total_amount") or 0.0
    
    # 1. Math Check (Floating point tolerance of 0.02)
    expected_total = subtotal + tax
    if abs(expected_total - total) > 0.02:
        issues.append(f"Math mismatch: Subtotal({subtotal}) + Tax({tax}) != Total({total})")
    
    # 2. Critical Fields Check
    if not data.get("vendor_details", {}).get("company_name"):
        issues.append("Missing Vendor Name")
    
    # Add review flag if issues found
    if issues:
        data["requires_human_review"] = True
        data["validation_errors"] = issues
    else:
        data["requires_human_review"] = False
        
    return data

## 4. Full Pipeline Test Run

In [5]:
def run_full_test(image_path):
    start_total = time.time()
    
    # 1. VLM Step
    print("Step 1: VLM Processing...")
    vlm_start = time.time()
    pipeline = PaddleOCRVL(
        vl_rec_backend="vllm-server",
        vl_rec_server_url=VLM_SERVER_URL,
        vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL"
    )
    results = pipeline.predict(image_path)
    raw_text = extract_and_combine_content(results)
    vlm_duration = time.time() - vlm_start
    
    # 2. Pre-processing
    print("Step 2: Pre-processing text...")
    cleaned_text = clean_ocr_text(raw_text)
    
    # 3. LLM Step
    print("Step 3: LLM Extraction...")
    llm_start = time.time()
    client = OpenAI(base_url=LLM_SERVER_URL, api_key="EMPTY")
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text to process:\n{cleaned_text}"}
        ],
        temperature=0.1,
        response_format={"type": "json_object"}
    )
    extracted_json = json.loads(response.choices[0].message.content)
    llm_duration = time.time() - llm_start
    
    # 4. Validation Gate
    print("Step 4: Validating results...")
    final_data = validate_extraction(extracted_json)
    
    total_duration = time.time() - start_total
    
    return {
        "final_json": final_data,
        "timings": {
            "vlm_sec": round(vlm_duration, 2),
            "llm_sec": round(llm_duration, 2),
            "total_sec": round(total_duration, 2)
        },
        "cleaned_text": cleaned_text
    }

# Execution (Uncomment if servers are running)
result = run_full_test(DEMO_IMAGE)
print(json.dumps(result, indent=2))

Step 1: VLM Processing...


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.5-0.9B', None)
[2026-04-14 15:16:58,027] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 15:16:58,038] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 15:16:58,040] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HT

Step 2: Pre-processing text...
Step 3: LLM Extraction...


[2026-04-14 15:17:05,420] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"


Step 4: Validating results...
{
  "final_json": {
    "document_details": {
      "document_type": "Invoice",
      "invoice_number": "12074",
      "invoice_date": "2024-07-19",
      "due_date": null
    },
    "vendor_details": {
      "company_name": "Wedding Photography",
      "person_name": null,
      "address": "123 Street Name Denver, CO 80205",
      "contact_info": "555-555-5555"
    },
    "client_details": {
      "company_name": null,
      "person_name": null,
      "address": null,
      "contact_info": null
    },
    "line_items": [
      {
        "description": "Wedding Ceremony Photos",
        "quantity": 1,
        "unit_price": 1500.0,
        "line_total": 1500.0
      },
      {
        "description": "Bride & Groom Portraits",
        "quantity": 1,
        "unit_price": 500.0,
        "line_total": 500.0
      },
      {
        "description": "Engagement Photoshoot",
        "quantity": 1,
        "unit_price": 300.0,
        "line_total": 300.0
      }
  